# Justification annotation — v2 pilot review

Checking the DeepSeek annotations against `src/prompts/justification_annotation_v2.txt`.

**What changed in v2.** The scheme stopped coding the *form* of the inference and
started coding the *type of information* the sentence reasons with:

| v1 | v2 |
|---|---|
| `Deduction` | `Mechanical` |
| `Consistency` | `ClaimComparison` |
| `Social` | `SocialJudgment` |
| — | `Uncertainty` (new) |
| `use` (used/discounted/mentioned) | **removed** |
| `rule_mentioned` | **removed** |

`Testimony`, `Behavioral`, `Payoff` and `Other` carry over unchanged.

**Same 40 justifications as v1**, by construction — the sample is drawn with a
fixed seed, so `pilot_v1/pilot_sample.jsonl` and `pilot_v2/pilot_sample.jsonl`
are byte-identical. Anything that differs between the two pilots is the scheme,
not the sample.

Verdicts are saved to `pilot_v2/pilot_review_verdicts.csv` on every click, so you
can stop and come back.


In [ ]:
# ============================================================
# Setup
# ============================================================

from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start=None, repo_name="masters_thesis_sdg"):
    current = (start or Path.cwd()).resolve()
    while True:
        if current.name == repo_name:
            return current
        if current.parent == current:
            raise FileNotFoundError(f"Could not find repo root {repo_name!r} above {Path.cwd()}")
        current = current.parent


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.pt_annotation.review_tools import (
    load_pilot, annotations_long, sentences_long,
    VerdictStore, score,
)
from src.pt_annotation.review_widget import ReviewApp

PILOT_DIR = REPO_ROOT / "results" / "justification_annotation" / "pilot_v2"

# The schema is read from the saved metadata, so this notebook also opens
# pilot_v1 unchanged if you point PILOT_DIR at it.
records, sample, schema = load_pilot(PILOT_DIR)
annotations = annotations_long(records)
sentences = sentences_long(records)

print(f"schema {schema.name}: {', '.join(schema.categories)}")
print(f"{len(records)} justifications, {len(sentences)} sentences, {len(annotations)} annotations")

## 1. Did the annotator follow the contract?

This has to hold before the labels are worth reading. The one that matters most
is **span exactness** — `evidence_span` must be a verbatim substring of its
sentence, or it cannot be aligned back to the text and the highlighting below
breaks.

`unexpected use field` is the v2-specific check: if it fires, the model is still
answering in the v1 scheme.

In [ ]:
# ============================================================
# 1. Contract compliance
# ============================================================

flagged = [r for r in records if r["metadata"].get("validation_flags")]
errored = [r for r in records if "error" in r["metadata"]]
truncated = [r for r in records if r["metadata"].get("finish_reason") not in (None, "stop")]

spans_missing = annotations[
    [s not in t for s, t in zip(annotations["evidence_span"].fillna(""), annotations["text"])]
]
bad_categories = annotations[~annotations["category"].isin(schema.categories)]
stale_use = annotations[annotations["use"].notna()] if not schema.has_use else annotations.iloc[0:0]

checks = pd.DataFrame([
    ("justifications returned",       f"{len(records)}/{len(sample)}"),
    ("API or parse errors",           len(errored)),
    ("validation flags raised",       len(flagged)),
    ("responses hit the token cap",   len(truncated)),
    ("evidence_span not verbatim",    len(spans_missing)),
    ("categories outside the scheme", len(bad_categories)),
    ("stale `use` field (v1 leakage)", len(stale_use)),
], columns=["check", "result"])

display(checks.style.hide(axis="index"))

if flagged:
    print("\nFlags raised:")
    for record in flagged:
        print(" ", record["metadata"]["justification_id"])
        for flag in record["metadata"]["validation_flags"]:
            print("   -", flag)
else:
    print("\nNo flags. Every response preserved the vote, the sentence ids and the")
    print("sentence text, stayed inside the v2 category vocabulary, and quoted spans")
    print("verbatim.")

## 2. What fired

Just enough to know which categories the pilot actually exercised. A category
with almost no instances hasn't been tested, so your review says nothing about
whether the prompt handles it — worth knowing before you start clicking.

`Uncertainty` is the one to watch: it's new in v2, and in v1 all four `Other`
annotations were absence-of-evidence reasoning, which is exactly what it was
added to catch.

In [ ]:
# ============================================================
# 2. What fired
# ============================================================

n_labelled = int(sentences["is_labelled"].sum())
print(f"labelled sentences   : {n_labelled}/{len(sentences)} "
      f"({100*n_labelled/len(sentences):.1f}%)")
print(f"multi-label sentences: {int((sentences['n_annotations'] > 1).sum())} "
      f"({100*(sentences['n_annotations'] > 1).mean():.1f}%)")
print()

counts = (
    annotations.groupby("category").size()
    .reindex(list(schema.categories)).fillna(0).astype(int)
    .to_frame("n")
)
counts["pct"] = (100 * counts["n"] / max(len(annotations), 1)).round(1)
counts["exercised"] = counts["n"] >= 5
display(counts)

other = annotations[annotations["category"].eq("Other")]
if len(other):
    print(f"{len(other)} Other annotations - if they name one recurring pattern, that is a")
    print("category the scheme is still missing:")
    for row in other.itertuples():
        print(f"  - {row.other_description}")

unlabelled = sentences[~sentences["is_labelled"]]
print(f"\n{len(unlabelled)} unlabelled sentences (the prompt allows this - check they")
print("really carry no reasoning, since a missed label looks identical):")
for row in unlabelled.itertuples():
    print(f"  [{row.model}] {row.text}")

## 3. Review

One justification at a time, evidence spans highlighted and colour-coded.

| verdict | meaning |
|---|---|
| `ok` | the annotation is right |
| `wrong category` | something is here, wrong label — set the correction |
| `wrong span` | right label, wrong span quoted |
| `spurious` | nothing should have been annotated here |

**"missed a label?"** catches the opposite failure — something that should have
been labelled and wasn't. There is no row to disagree with in that case, so this
is the only place it can be recorded.

**Mark all ok** accepts every not-yet-judged annotation on the current
justification and never overwrites a verdict you already set.

The `use` controls are gone: v2 has no such field, so the widget drops them
rather than showing dead dropdowns.

In [ ]:
# ============================================================
# 3. Reviewer
# ============================================================

store = VerdictStore(
    PILOT_DIR / "pilot_review_verdicts.csv",
    PILOT_DIR / "pilot_review_missed.csv",
)

app = ReviewApp(records, store, schema, sample)
app.display()

## 4. Result

Re-run after reviewing — it reads the saved verdicts, so it reflects partial
progress.

This is an **error rate against your adjudication**, not a chance-corrected
agreement coefficient: you reviewed the model's labels rather than coding blind,
so the two label sets aren't independent.

In [ ]:
# ============================================================
# 4. Score and findings
# ============================================================

store = VerdictStore(
    PILOT_DIR / "pilot_review_verdicts.csv",
    PILOT_DIR / "pilot_review_missed.csv",
)

per_category, summary = score(store.verdicts, annotations, schema)

if not summary:
    print("No verdicts recorded yet - work through section 3 first.")
else:
    print(f"reviewed    : {summary['n_reviewed']}/{summary['n_total']} annotations")
    print(f"accept rate : {summary['accept_rate']:.1f}%")
    print(f"verdicts    : {summary['verdict_counts']}")
    print(f"missed      : {len(store.missed)} sentences flagged as missing a label")
    print()
    display(per_category)

    disagreements = store.verdicts[
        store.verdicts["verdict"].ne("") & store.verdicts["verdict"].ne("ok")
    ]
    if not disagreements.empty:
        print("Disagreements:")
        display(disagreements)

        confusion = (
            disagreements[disagreements["corrected_category"].ne("")]
            .groupby(["category", "corrected_category"]).size()
            .rename("n").reset_index().sort_values("n", ascending=False)
        )
        if len(confusion):
            print("Boundaries being crossed (annotator said -> you said):")
            display(confusion)

    if not store.missed.empty:
        print("Sentences flagged as missing a label:")
        display(store.missed)